# RAG

## 프로젝트 개요
B2G 입찰 컨설팅 스타트업 **입찰메이트**의 사내 RAG 시스템.
100개의 RFP(제안요청서) 문서에서 핵심 정보를 추출하고, 질의응답을 제공한다.

## 파이프라인 구조
```
[원본 HWP/PDF] → [Parser] → [Retriever] → [LangGraph] → [답변]
     │               │             │              │
     │          텍스트 추출    벡터 검색     Self-Corrective RAG
     │          필드 추출     Chroma DB      답변 생성 + 출처 인용
     │          청킹          메타 필터      관련성 평가 + 재검색
```

## 모듈 구성
| 모듈 | 역할 |
| :--- | :--- |
| **Parser** | HWP/HWPX/PDF → 텍스트 추출, 정제, 청킹 |
| **Retriever** | 임베딩, Chroma 적재, 유사도 검색 |
| **Prompt** | 답변 생성 / 관련성 평가 / 질문 재작성 프롬프트 |
| **LangGraph** | Self-Corrective RAG 그래프 오케스트레이션 |

## 1. 환경 설정

In [1]:
# 필요 라이브러리 설치 (최초 1회)
# !pip install python-hwpx pyhwp lxml pymupdf4llm
# !pip install langchain langchain-openai langchain-chroma langchain-text-splitters
# !pip install langgraph chromadb pandas openpyxl python-dotenv

In [2]:
import os
import re
import time
import zipfile
from pathlib import Path
from typing import List, Optional

import pandas as pd
import pymupdf4llm
from dotenv import load_dotenv
from lxml import etree
from pydantic import BaseModel, Field

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langgraph.graph import END, StateGraph
from typing import TypedDict

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


In [3]:
# .env 로드 및 경로 설정
PROJECT_ROOT = Path(os.getcwd()).parent
env_path = PROJECT_ROOT / ".env"
load_dotenv(dotenv_path=env_path)

DATA_DIR = PROJECT_ROOT / "원본 데이터"
FILES_DIR = DATA_DIR / "files"
CSV_PATH = DATA_DIR / "data_list.csv"
PERSIST_DIR = str(PROJECT_ROOT / "chroma_db")

assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY가 없습니다. .env 파일을 확인해주세요."
)

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f".env 로드: {env_path.exists()}")
print(f"CSV 존재: {CSV_PATH.exists()}")
print(f"OPENAI_API_KEY: 설정됨")

프로젝트 루트: c:\Users\User\Desktop\중급 프로젝트
.env 로드: True
CSV 존재: True
OPENAI_API_KEY: 설정됨


---
## 2. Parser — 문서 파싱

원본 RFP 문서(HWP/HWPX/PDF)를 읽어 구조화된 텍스트로 변환한다.
```
[원본 파일] → [텍스트 추출] → [정제] → [필드 추출] → [청킹] → [Document 리스트]
```

In [4]:
# CSV 메타데이터 로드
df_raw = pd.read_csv(CSV_PATH, encoding="utf-8")

print(f"전체 문서 수: {len(df_raw)}")
print(f"파일형식 분포:\n{df_raw['파일형식'].value_counts()}")
print(f"\n텍스트 길이 통계:\n{df_raw['텍스트'].str.len().describe()}")

전체 문서 수: 100
파일형식 분포:
파일형식
hwp    96
pdf     4
Name: count, dtype: int64

텍스트 길이 통계:
count      100.000000
mean      3843.530000
std       3692.593749
min         89.000000
25%       1198.000000
50%       2583.000000
75%       5842.000000
max      18335.000000
Name: 텍스트, dtype: float64


In [5]:
# --- HWPX 파서 (ZIP 내부 XML) ---

def extract_text_from_hwpx(file_path: str) -> str:
    """HWPX 파일에서 본문 텍스트를 추출한다."""
    texts = []
    try:
        with zipfile.ZipFile(file_path, "r") as zf:
            section_files = sorted(
                name for name in zf.namelist()
                if "section" in name.lower() and name.endswith(".xml")
            )
            for sf in section_files:
                root = etree.fromstring(zf.read(sf))
                for node in root.iter():
                    if node.text and node.text.strip():
                        texts.append(node.text.strip())
    except zipfile.BadZipFile:
        return ""
    except Exception as e:
        return f"[HWPX 파싱 오류: {e}]"
    return "\n".join(texts)


# --- HWP 5.x 파서 (레거시 바이너리) ---

def extract_text_from_hwp_library(file_path: str) -> str:
    """python-hwpx 라이브러리로 HWP 파일 텍스트를 추출한다."""
    try:
        from hwpx import HWPXFile
        return HWPXFile(file_path).get_text()
    except ImportError:
        return "[python-hwpx 미설치]"
    except Exception as e:
        return f"[HWP 파싱 오류: {e}]"


# --- PDF 파서 (pymupdf4llm) ---

def extract_text_from_pdf(file_path: str) -> str:
    """PDF 파일에서 pymupdf4llm으로 마크다운 텍스트를 추출한다."""
    try:
        return pymupdf4llm.to_markdown(file_path)
    except Exception as e:
        return f"[PDF 파싱 오류: {e}]"


print("파서 함수 정의 완료: HWPX, HWP, PDF")

파서 함수 정의 완료: HWPX, HWP, PDF


In [6]:
# --- 텍스트 정제 ---

PAGE_NUMBER_PATTERNS = [
    r"^\s*-\s*\d+\s*-\s*$",
    r"^\s*--\s*\d+\s*--\s*$",
    r"^\s*\(\d+\)\s*$",
    r"^\s*\d+\s*/\s*\d+\s*$",
    r"^\s*페이지\s*\d+\s*$",
]
HEADER_FOOTER_PATTERNS = [
    r"^\s*제안요청서\s*$",
    r"^\s*- \d+ -\s*$",
]


def clean_text(raw_text: str) -> str:
    """페이지 번호/머리글 제거, 공백 정규화."""
    if not raw_text or not raw_text.strip():
        return ""
    lines = raw_text.split("\n")
    all_patterns = PAGE_NUMBER_PATTERNS + HEADER_FOOTER_PATTERNS
    compiled = [re.compile(p, re.MULTILINE) for p in all_patterns]
    cleaned_lines = []
    for line in lines:
        if any(p.match(line) for p in compiled):
            continue
        cleaned = re.sub(r"[ \t]+", " ", line).strip()
        if cleaned:
            cleaned_lines.append(cleaned)
    result = "\n".join(cleaned_lines)
    return re.sub(r"\n{3,}", "\n\n", result)


print("clean_text 정의 완료")

clean_text 정의 완료


In [7]:
# --- 핵심 정보 필드 추출 ---

BID_TYPE_KEYWORDS = [
    "협상에 의한 계약", "제한경쟁입찰", "일반경쟁입찰",
    "수의계약", "지명경쟁입찰", "협상에의한계약",
    "제한경쟁", "일반경쟁",
]


def extract_bid_type(text: str) -> str:
    for kw in BID_TYPE_KEYWORDS:
        if kw in text:
            return kw
    return ""


def extract_qualification(text: str) -> str:
    for pat in [
        r"참가\s*자격[^\n]*\n((?:.*\n){1,10})",
        r"입찰\s*참가\s*자격[^\n]*\n((?:.*\n){1,10})",
        r"참여\s*자격[^\n]*\n((?:.*\n){1,10})",
    ]:
        m = re.search(pat, text)
        if m:
            return m.group(1).strip()
    return ""


def extract_fields(row: pd.Series, cleaned_text: str) -> dict:
    """CSV 메타 + 텍스트 분석으로 핵심 필드를 추출한다."""
    budget_raw = row.get("사업 금액")
    if pd.notna(budget_raw):
        bv = int(budget_raw)
        budget_str = (
            f"{bv / 1e8:.1f}억원" if bv >= 1e8
            else f"{bv / 1e4:.0f}만원" if bv >= 1e4
            else f"{bv:,}원"
        )
    else:
        budget_str = ""
    deadline = str(row.get("입찰 참여 마감일", ""))
    if deadline == "nan":
        deadline = ""
    return {
        "사업명": str(row.get("사업명", "")),
        "발주기관": str(row.get("발주 기관", "")),
        "사업예산": budget_str,
        "입찰방식": extract_bid_type(cleaned_text),
        "제출기한": deadline,
        "참가자격": extract_qualification(cleaned_text),
    }


print("필드 추출 함수 정의 완료")

필드 추출 함수 정의 완료


In [8]:
# --- 청킹 ---

CHUNK_SIZE = 800   # 1000→800: 짧은 문서에서도 최소 1개 이상 청크 생성
CHUNK_OVERLAP = 200
SEPARATORS = ["\n\n", "\n", ". ", "다. ", "요. ", "함. ", " "]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=SEPARATORS,
    length_function=len,
)


def create_chunks(cleaned_text: str, metadata: dict) -> list[Document]:
    """텍스트를 청크로 분할하고 메타데이터를 부착한다."""
    if not cleaned_text.strip():
        return []
    raw_chunks = text_splitter.split_text(cleaned_text)
    chunks = [
        Document(
            page_content=chunk,
            metadata={**metadata, "chunk_index": idx, "chunk_total": len(raw_chunks)},
        )
        for idx, chunk in enumerate(raw_chunks)
    ]
    return chunks


def create_meta_summary_chunk(metadata: dict) -> Document:
    """CSV 메타데이터로 요약 청크를 생성한다.
    텍스트에 누락된 예산/기한/기관 정보를 검색 가능하게 만든다.
    """
    parts = []
    if metadata.get("사업명"):
        parts.append(f"사업명: {metadata['사업명']}")
    if metadata.get("발주기관"):
        parts.append(f"발주기관: {metadata['발주기관']}")
    if metadata.get("사업예산"):
        parts.append(f"사업예산: {metadata['사업예산']}")
    if metadata.get("입찰방식"):
        parts.append(f"입찰방식: {metadata['입찰방식']}")
    if metadata.get("제출기한"):
        parts.append(f"제출기한: {metadata['제출기한']}")
    if metadata.get("참가자격"):
        parts.append(f"참가자격: {metadata['참가자격']}")
    if metadata.get("사업요약"):
        parts.append(f"사업요약: {metadata['사업요약']}")
    if metadata.get("공고번호"):
        parts.append(f"공고번호: {metadata['공고번호']}")
    if not parts:
        return None
    content = "\n".join(parts)
    meta = {**metadata, "chunk_index": -1, "chunk_total": 0, "is_meta_summary": "true"}
    return Document(page_content=content, metadata=meta)


print(f"청킹 설정: size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")
print("메타 요약 청크 함수 정의 완료")

청킹 설정: size=800, overlap=200
메타 요약 청크 함수 정의 완료


In [9]:
# --- 통합 파싱 파이프라인 ---

def parse_single_document(row: pd.Series, files_dir: Path) -> dict:
    """단일 RFP 문서를 파싱한다."""
    file_name = str(row["파일명"])
    errors = []

    # Step 1: 텍스트 추출 (CSV 우선)
    raw_text = row.get("텍스트")
    if pd.isna(raw_text) or not str(raw_text).strip() or len(str(raw_text).strip()) < 50:
        file_path = files_dir / file_name
        if file_path.exists():
            suffix = file_path.suffix.lower()
            if suffix == ".pdf":
                raw_text = extract_text_from_pdf(str(file_path))
            elif suffix == ".hwpx":
                raw_text = extract_text_from_hwpx(str(file_path))
            elif suffix in (".docx", ".doc"):
                raw_text = extract_text_from_pdf(str(file_path))
            else:
                raw_text = extract_text_from_hwp_library(str(file_path))
        else:
            errors.append(f"파일 없음: {file_name}")
            raw_text = ""
    raw_text = str(raw_text)

    # 파싱 오류 태그가 포함된 경우 에러 기록
    if raw_text.startswith("[") and "오류" in raw_text:
        errors.append(raw_text)
        raw_text = ""

    # Step 2: 정제
    cleaned_text = clean_text(raw_text)
    if not cleaned_text:
        errors.append(f"텍스트 추출 실패: {file_name}")

    # Step 3: 필드 추출
    try:
        metadata = extract_fields(row, cleaned_text)
    except (ValueError, TypeError) as e:
        errors.append(f"필드 추출 오류 ({file_name}): {e}")
        metadata = {
            "사업명": str(row.get("사업명", "")),
            "발주기관": str(row.get("발주 기관", "")),
            "사업예산": "", "입찰방식": "", "제출기한": "", "참가자격": "",
        }
    metadata["source"] = file_name
    metadata["공고번호"] = str(row.get("공고 번호", ""))
    metadata["공개일자"] = str(row.get("공개 일자", ""))
    metadata["사업요약"] = str(row.get("사업 요약", ""))[:200]

    # Step 4: 청킹
    chunks = create_chunks(cleaned_text, metadata)

    # Step 5: 메타 요약 청크 추가 (CSV 메타데이터 기반)
    meta_chunk = create_meta_summary_chunk(metadata)
    if meta_chunk:
        chunks.append(meta_chunk)

    if not chunks and cleaned_text:
        errors.append(f"청킹 결과 없음: {file_name}")

    return {"file_name": file_name, "chunks": chunks, "metadata": metadata, "errors": errors}


def parse_all_documents(df: pd.DataFrame, files_dir: Path):
    """전체 RFP 문서를 배치 파싱한다."""
    all_chunks, all_metadata, all_errors = [], [], []
    skipped = 0
    for idx, row in df.iterrows():
        result = parse_single_document(row, files_dir)
        all_chunks.extend(result["chunks"])
        all_metadata.append(result["metadata"])
        all_errors.extend(result["errors"])
        if not result["chunks"]:
            skipped += 1
        if (idx + 1) % 20 == 0:
            print(f"  파싱: {idx + 1}/{len(df)} 문서")
    if skipped:
        print(f"  [주의] 청크 0개 문서: {skipped}건")
    return all_chunks, all_metadata, all_errors


print("파싱 파이프라인 정의 완료 (메타 요약 청크 포함)")

파싱 파이프라인 정의 완료 (메타 요약 청크 포함)


In [10]:
# === 전체 파싱 실행 ===
print("=== RFP 문서 파싱 시작 ===")
print(f"대상 문서 수: {len(df_raw)}\n")

all_chunks, all_metadata, all_errors = parse_all_documents(df_raw, FILES_DIR)

print(f"\n=== 파싱 결과 ===")
print(f"총 청크 수: {len(all_chunks)}")
print(f"문서당 평균: {len(all_chunks) / max(len(df_raw), 1):.1f}개")
print(f"에러 수: {len(all_errors)}")
if all_errors:
    for err in all_errors[:10]:
        print(f"  - {err}")

=== RFP 문서 파싱 시작 ===
대상 문서 수: 100

  파싱: 20/100 문서
  파싱: 40/100 문서
  파싱: 60/100 문서
  파싱: 80/100 문서
  파싱: 100/100 문서

=== 파싱 결과 ===
총 청크 수: 671
문서당 평균: 6.7개
에러 수: 0


In [11]:
# 파싱 결과 검증
chunk_lengths = [len(c.page_content) for c in all_chunks]
df_meta = pd.DataFrame(all_metadata)

print("=== 청크 길이 통계 ===")
if chunk_lengths:
    print(f"  총: {len(chunk_lengths)}, 최소: {min(chunk_lengths)}, 최대: {max(chunk_lengths)}")
    print(f"  평균: {sum(chunk_lengths)/len(chunk_lengths):.0f}")

print("\n=== 필드 추출 현황 ===")
for col in ["사업명", "발주기관", "사업예산", "입찰방식", "제출기한", "참가자격"]:
    n = df_meta[col].apply(lambda x: bool(x and str(x).strip())).sum()
    print(f"  {col}: {n}/{len(df_meta)} ({n/len(df_meta)*100:.0f}%)")

=== 청크 길이 통계 ===
  총: 671, 최소: 66, 최대: 1230
  평균: 679

=== 필드 추출 현황 ===
  사업명: 100/100 (100%)
  발주기관: 100/100 (100%)
  사업예산: 99/100 (99%)
  입찰방식: 63/100 (63%)
  제출기한: 92/100 (92%)
  참가자격: 46/100 (46%)


---
## 3. Retriever — 문서 검색

Parser의 청크를 벡터 DB에 적재하고, 유사도 기반으로 검색한다.
```
[all_chunks] → [Embedding] → [Chroma 적재] → [유사도 검색] → [GraphState 반환]
```

In [12]:
# --- 검색 설정 ---
EMBEDDING_MODEL = "text-embedding-3-small"
COLLECTION_NAME = "bidmate_rfp"
DEFAULT_K = 5
DEFAULT_SEARCH_TYPE = "similarity"
SCORE_THRESHOLD = 0.2  # 0.3에서 하향 (한국어 RFP 임베딩 스코어가 0.18~0.28 분포)


def create_embeddings(model_name: str = EMBEDDING_MODEL) -> OpenAIEmbeddings:
    """임베딩 모델을 생성한다. Vertex AI 교체 시 이 함수만 수정."""
    return OpenAIEmbeddings(model=model_name)


embeddings = create_embeddings()

# 동작 확인
test_vec = embeddings.embed_query("테스트")
print(f"임베딩 모델: {EMBEDDING_MODEL}")
print(f"임베딩 차원: {len(test_vec)}")
print(f"유사도 임계값: {SCORE_THRESHOLD}")

임베딩 모델: text-embedding-3-small
임베딩 차원: 1536
유사도 임계값: 0.2


In [13]:
# --- 벡터스토어 생성/로드 ---
import shutil
import gc


def create_vectorstore(documents, embedding_fn, collection_name=COLLECTION_NAME, persist_dir=PERSIST_DIR):
    """청크를 Chroma에 적재한다."""
    if not documents:
        raise ValueError("적재할 Document가 없습니다. Parser 결과를 확인하세요.")
    return Chroma.from_documents(
        documents=documents,
        embedding=embedding_fn,
        collection_name=collection_name,
        persist_directory=persist_dir,
    )


def load_vectorstore(embedding_fn, collection_name=COLLECTION_NAME, persist_dir=PERSIST_DIR):
    """영속화된 Chroma를 로드한다."""
    if not Path(persist_dir).exists():
        return None
    try:
        store = Chroma(
            collection_name=collection_name,
            embedding_function=embedding_fn,
            persist_directory=persist_dir,
        )
        store._collection.count()
        return store
    except Exception:
        return None


def _safe_remove_db(persist_dir, existing_store=None):
    """Windows 호환: Chroma 연결 해제 후 DB 디렉토리를 삭제한다."""
    if existing_store is not None:
        try:
            existing_store._client = None
            existing_store._collection = None
        except Exception:
            pass
        del existing_store
        gc.collect()
    if Path(persist_dir).exists():
        shutil.rmtree(persist_dir, ignore_errors=True)


# 청크 수가 변경되었으면 기존 DB 삭제 후 재생성
FORCE_REBUILD = False  # True로 변경하면 강제 재생성
existing = load_vectorstore(embeddings)

if existing and not FORCE_REBUILD:
    old_count = existing._collection.count()
    if old_count == len(all_chunks):
        vectorstore = existing
        print(f"기존 벡터스토어 로드: {old_count}개 벡터")
    else:
        print(f"청크 수 불일치 (DB: {old_count}, 현재: {len(all_chunks)}) → 재생성")
        _safe_remove_db(PERSIST_DIR, existing)
        existing = None
        vectorstore = create_vectorstore(all_chunks, embeddings)
        print(f"적재 완료: {vectorstore._collection.count()}개 벡터")
else:
    if FORCE_REBUILD and Path(PERSIST_DIR).exists():
        _safe_remove_db(PERSIST_DIR, existing)
        existing = None
        print("기존 DB 삭제 (FORCE_REBUILD)")
    assert len(all_chunks) > 0, "all_chunks가 비어있습니다."
    print(f"새 벡터스토어 생성 중... ({len(all_chunks)}개 청크)")
    vectorstore = create_vectorstore(all_chunks, embeddings)
    print(f"적재 완료: {vectorstore._collection.count()}개 벡터")

print(f"영속화 경로: {PERSIST_DIR}")

청크 수 불일치 (DB: 1784, 현재: 671) → 재생성
적재 완료: 2455개 벡터
영속화 경로: c:\Users\User\Desktop\중급 프로젝트\chroma_db


In [14]:
# --- 검색 함수 ---

MAX_CONTEXT_CHARS = 4000  # generate에 전달할 컨텍스트 최대 길이 (3000→4000 확대)


def search_similarity(query, k=DEFAULT_K, score_threshold=SCORE_THRESHOLD):
    """유사도 검색. 임계값 미달 시 top-k 폴백."""
    results = vectorstore.similarity_search_with_relevance_scores(query, k=k)
    filtered = [(doc, score) for doc, score in results if score >= score_threshold]
    # 폴백: 임계값 통과 결과가 없으면 상위 k개 그대로 반환
    if not filtered and results:
        return results
    return filtered


def search_mmr(query, k=DEFAULT_K, fetch_k=20, lambda_mult=0.5):
    """MMR 검색: 유사도 + 다양성."""
    return vectorstore.max_marginal_relevance_search(
        query, k=k, fetch_k=fetch_k, lambda_mult=lambda_mult
    )


def search_with_filter(query, filters, k=DEFAULT_K):
    """메타데이터 필터 적용 검색."""
    where = {}
    if filters.get("기관"):
        where["발주기관"] = filters["기관"]
    if filters.get("파일"):
        where["source"] = filters["파일"]
    search_kwargs = {"k": k}
    if where:
        search_kwargs["filter"] = where
    return vectorstore.similarity_search(query, **search_kwargs)


def _truncate_contexts(contexts: list[str], max_chars: int = MAX_CONTEXT_CHARS) -> list[str]:
    """총 문자 수가 max_chars를 넘지 않도록 컨텍스트를 잘라낸다."""
    result = []
    total = 0
    for ctx in contexts:
        if total + len(ctx) > max_chars:
            remaining = max_chars - total
            if remaining > 200:
                result.append(ctx[:remaining] + "...")
            break
        result.append(ctx)
        total += len(ctx)
    return result


def retrieve_for_graph(question, filters=None, search_type=DEFAULT_SEARCH_TYPE, k=DEFAULT_K):
    """GraphState 호환 통합 검색 함수."""
    start_ts = time.time()
    error_msg = None
    try:
        if filters:
            docs = search_with_filter(question, filters, k=k)
        elif search_type == "mmr":
            docs = search_mmr(question, k=k)
        else:
            scored = search_similarity(question, k=k)
            docs = [doc for doc, _ in scored]
        contexts = [doc.page_content for doc in docs]
        citations = [doc.metadata.get("source", "N/A") for doc in docs]
    except Exception as e:
        error_msg = f"검색 오류: {e}"
        contexts, citations = [], []

    result = {
        "retrieved_contexts": contexts,
        "citations_used": citations,
        "latency": round(time.time() - start_ts, 4),
    }
    if error_msg:
        result["errors"] = [error_msg]
    return result


print("검색 함수 정의 완료: similarity(+폴백), mmr, filter, retrieve_for_graph")

검색 함수 정의 완료: similarity(+폴백), mmr, filter, retrieve_for_graph


In [15]:
# Retriever 동작 테스트
test_q = "이 사업의 총 예산은 얼마인가요?"
test_result = retrieve_for_graph(test_q)

print(f"=== Retriever 테스트 ===")
print(f"질문: {test_q}")
print(f"검색 결과: {len(test_result['retrieved_contexts'])}개")
print(f"응답 시간: {test_result['latency']}s")
for i, (ctx, src) in enumerate(zip(test_result["retrieved_contexts"][:3], test_result["citations_used"][:3])):
    print(f"\n--- 결과 {i+1} [{src}] ---")
    print(f"{ctx[:200]}...")

=== Retriever 테스트 ===
질문: 이 사업의 총 예산은 얼마인가요?
검색 결과: 5개
응답 시간: 0.3283s

--- 결과 1 [한국농수산식품유통공사_농산물가격안정기금 정부예산회계연계시스템 .hwp] ---
7. 기타 - 53
Ⅵ. 제안서 작성요령 - 55
1. 제안서의 효력 - 55
2. 제안서 작성 시 유의사항 - 55
3. 세부 작성지침 - 58
[붙임] - 61
[붙임 1호 서식] 업체 일반현황 - 62
[붙임 2호 서식] 수행기관 경영상태 - 63
[붙임 3호 서식] 조직현황 - 64
[붙임 4호 서식] 본 사업 추진별 업무분장 등 요약 - 65
[붙...

--- 결과 2 [한국농수산식품유통공사_농산물가격안정기금 정부예산회계연계시스템 .hwp] ---
7. 기타 - 53
Ⅵ. 제안서 작성요령 - 55
1. 제안서의 효력 - 55
2. 제안서 작성 시 유의사항 - 55
3. 세부 작성지침 - 58
[붙임] - 61
[붙임 1호 서식] 업체 일반현황 - 62
[붙임 2호 서식] 수행기관 경영상태 - 63
[붙임 3호 서식] 조직현황 - 64
[붙임 4호 서식] 본 사업 추진별 업무분장 등 요약 - 65
[붙...

--- 결과 3 [한국농수산식품유통공사_농산물가격안정기금 정부예산회계연계시스템 .hwp] ---
7. 기타 - 53
Ⅵ. 제안서 작성요령 - 55
1. 제안서의 효력 - 55
2. 제안서 작성 시 유의사항 - 55
3. 세부 작성지침 - 58
[붙임] - 61
[붙임 1호 서식] 업체 일반현황 - 62
[붙임 2호 서식] 수행기관 경영상태 - 63
[붙임 3호 서식] 조직현황 - 64
[붙임 4호 서식] 본 사업 추진별 업무분장 등 요약 - 65
[붙...


---
## 4. Prompt — 프롬프트 엔지니어링

3가지 LCEL 체인을 구성한다:
- **Generate**: 검색 컨텍스트 기반 답변 생성
- **Grade**: 문서 관련성 이진 판별
- **Rewrite**: 검색 최적화 질문 재작성

In [16]:
# === 프롬프트 템플릿 ===

GENERATE_SYSTEM = """너는 B2G(정부/공공기관) 입찰 컨설팅 전문 AI 비서 '입찰메이트'야.

[핵심 규칙]
1. 반드시 아래 제공된 RFP 문서 컨텍스트에 기반하여 답변해.
2. 컨텍스트에 없는 내용은 절대 추측하지 말고, "제공된 문서에서 해당 정보를 확인할 수 없습니다."라고 답변해.
3. 답변 말미에 참고한 문서 출처를 반드시 명시해.
4. 금액, 일자, 자격 요건 등 핵심 수치는 원문 그대로 인용해.
5. 여러 문서에 걸쳐 정보가 있으면 통합하여 정리해.

[답변 형식]
- 핵심 내용을 먼저 요약
- 세부 사항은 불릿 포인트로 정리
- 마지막에 [출처] 표기
"""

GENERATE_HUMAN = """다음 RFP 문서 내용을 참고하여 질문에 답변해주세요.

=== 참고 문서 ===
{context}

=== 문서 출처 ===
{citations}

=== 질문 ===
{question}
"""

GRADE_SYSTEM = """너는 RFP(제안요청서) 문서 검수 전문가야.
아래 문서 내용이 질문에 답하는 데 필요한 핵심 정보를 담고 있는지 판별해줘.

[판별 기준]
- 질문의 핵심 주제와 관련된 구체적 정보가 있으면 'yes'
- 질문과 무관하거나 답변에 필요한 정보가 없으면 'no'
- 부분적으로 관련있더라도 핵심 정보가 있으면 'yes'

반드시 'yes' 또는 'no'로만 답변해."""

GRADE_HUMAN = """질문: {question}

문서 내용: {context}"""

REWRITE_SYSTEM = """너는 B2G 입찰 문서 검색 최적화 전문가야.
사용자의 질문을 벡터 검색에 최적화된 형태로 재작성해줘.

[규칙]
1. 30자 이내로 짧게 재작성해. 길면 검색 품질이 떨어져.
2. RFP 도메인 동의어를 1-2개 추가해. (예: 예산→추정가격, 마감→제출기한)
3. 재작성된 질문만 출력해."""

REWRITE_HUMAN = """원래 질문: {question}

짧고 핵심적인 검색 질문으로 재작성해."""

print("프롬프트 템플릿 정의 완료: GENERATE, GRADE, REWRITE")

프롬프트 템플릿 정의 완료: GENERATE, GRADE, REWRITE


In [17]:
# === 출력 스키마 + LLM + LCEL 체인 ===

class GradeAnswer(BaseModel):
    """문서 관련성 판별 결과."""
    binary_score: str = Field(description="관련 있으면 'yes', 없으면 'no'")

# 용도별 LLM
llm_generate = ChatOpenAI(model="gpt-5-mini", temperature=0.1)
llm_grade    = ChatOpenAI(model="gpt-5-mini", temperature=0)
llm_rewrite  = ChatOpenAI(model="gpt-5-mini", temperature=0.3)

# Generate Chain
generate_prompt = ChatPromptTemplate.from_messages([
    ("system", GENERATE_SYSTEM), ("human", GENERATE_HUMAN),
])
generate_chain = generate_prompt | llm_generate | StrOutputParser()

# Grade Chain (Structured Output)
grade_prompt = ChatPromptTemplate.from_messages([
    ("system", GRADE_SYSTEM), ("human", GRADE_HUMAN),
])
grade_chain = grade_prompt | llm_grade.with_structured_output(GradeAnswer)

# Rewrite Chain
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", REWRITE_SYSTEM), ("human", REWRITE_HUMAN),
])
rewrite_chain = rewrite_prompt | llm_rewrite | StrOutputParser()

print("LCEL 체인 구성 완료")
print(f"  generate_chain  입력: {generate_prompt.input_variables}")
print(f"  grade_chain     입력: {grade_prompt.input_variables}")
print(f"  rewrite_chain   입력: {rewrite_prompt.input_variables}")

LCEL 체인 구성 완료
  generate_chain  입력: ['citations', 'context', 'question']
  grade_chain     입력: ['context', 'question']
  rewrite_chain   입력: ['question']


---
## 5. LangGraph — Self-Corrective RAG

검색 → 관련성 평가 → 조건부 답변 생성/재검색 흐름을 구성한다.
```
retrieve → grade_documents → (조건부)
                                ├─ 관련성 충분 → generate → END
                                └─ 관련성 부족 → transform_query → retrieve (재검색)
```

In [18]:
# === GraphState 정의 ===

class GraphState(TypedDict):
    """RAG 파이프라인의 공유 상태."""
    question: str              # 사용자 질문
    chat_history: List[str]    # 대화 기록
    filters: dict              # 검색 필터
    retrieved_contexts: List[str]  # 검색된 문서 원문
    final_answer: str          # 최종 답변
    citations_used: List[str]  # 출처 (파일명)
    errors: List[str]          # 에러 메시지
    relevance_score: float     # 관련성 점수
    retry_count: int           # 재시도 횟수
    latency: float             # 검색 소요 시간
    hit_position: int          # 정답 문서 순위 (평가용)

print("GraphState 정의 완료")
print(f"  필드: {list(GraphState.__annotations__.keys())}")

GraphState 정의 완료
  필드: ['question', 'chat_history', 'filters', 'retrieved_contexts', 'final_answer', 'citations_used', 'errors', 'relevance_score', 'retry_count', 'latency', 'hit_position']


In [19]:
# === LangGraph 노드 함수 ===

MAX_RETRIES = 2


def retrieve(state: GraphState) -> dict:
    """벡터스토어에서 관련 문서를 검색한다."""
    question = state.get("question", "")
    filters = state.get("filters", {})
    result = retrieve_for_graph(question, filters)
    return result


def generate(state: GraphState) -> dict:
    """검색된 컨텍스트 기반으로 답변을 생성한다."""
    question = state.get("question", "")
    contexts = state.get("retrieved_contexts", [])
    citations = state.get("citations_used", [])

    if not contexts:
        return {
            "final_answer": "관련 문서를 찾을 수 없습니다.",
            "errors": [*state.get("errors", []), "컨텍스트 없음"],
        }

    # 토큰 효율: 컨텍스트 총량 제한
    truncated = _truncate_contexts(contexts)
    context_str = "\n\n---\n\n".join(
        f"[문서 {i+1}]\n{ctx}" for i, ctx in enumerate(truncated)
    )
    citations_str = ", ".join(dict.fromkeys(citations))

    try:
        answer = generate_chain.invoke({
            "context": context_str,
            "citations": citations_str,
            "question": question,
        })
    except Exception as e:
        return {
            "final_answer": "답변 생성 중 오류가 발생했습니다.",
            "errors": [*state.get("errors", []), f"LLM 호출 실패: {e}"],
        }
    return {"final_answer": answer}


def grade_documents(state: GraphState) -> dict:
    """검색된 문서의 관련성을 LLM으로 평가한다."""
    question = state.get("question", "")
    documents = state.get("retrieved_contexts", [])

    if not documents:
        print("--- 검수: 문서 없음 ---")
        return {"relevance_score": 0.0}

    relevant_count = 0
    for i, doc in enumerate(documents):
        try:
            result = grade_chain.invoke({"question": question, "context": doc})
            grade = result.binary_score
        except Exception:
            grade = "no"
        if grade == "yes":
            relevant_count += 1
        print(f"  문서 {i+1}: {grade}")

    score = relevant_count / len(documents)
    print(f"--- 검수 완료: {relevant_count}/{len(documents)} 관련, 점수 {score:.2f} ---")
    return {"relevance_score": score}


def transform_query(state: GraphState) -> dict:
    """질문을 재작성하여 검색 품질을 높인다."""
    original = state.get("question", "")
    retry_count = state.get("retry_count", 0) + 1

    try:
        rewritten = rewrite_chain.invoke({"question": original})
    except Exception:
        rewritten = original  # 실패 시 원래 질문 유지

    print(f"--- 질문 재작성 (retry {retry_count}) ---")
    print(f"  원래: {original}")
    print(f"  변환: {rewritten}")
    return {"question": rewritten, "retry_count": retry_count}


def decide_to_generate(state: GraphState) -> str:
    """관련성에 따라 다음 노드를 결정한다."""
    if state.get("retry_count", 0) >= MAX_RETRIES:
        print(f"--- 최대 재시도({MAX_RETRIES}회) 도달 → 답변 생성 ---")
        return "generate"
    if state.get("relevance_score", 0) >= 0.5:
        print("--- 관련성 충분 → 답변 생성 ---")
        return "generate"
    print("--- 관련성 부족 → 질문 재작성 ---")
    return "transform_query"


print("노드 함수 정의 완료: retrieve, generate, grade_documents, transform_query")

노드 함수 정의 완료: retrieve, generate, grade_documents, transform_query


In [20]:
# === MVP 단순 RAG 그래프 ===

def build_simple_graph():
    """retrieve → generate → END"""
    graph = StateGraph(GraphState)
    graph.add_node("retrieve", retrieve)
    graph.add_node("generate", generate)
    graph.set_entry_point("retrieve")
    graph.add_edge("retrieve", "generate")
    graph.add_edge("generate", END)
    return graph.compile()

simple_graph = build_simple_graph()
print("Simple RAG 그래프 빌드 완료")

Simple RAG 그래프 빌드 완료


In [21]:
# === Self-Corrective RAG 그래프 ===

def build_self_corrective_graph():
    """retrieve → grade → (generate | transform_query → retrieve)"""
    graph = StateGraph(GraphState)
    graph.add_node("retrieve", retrieve)
    graph.add_node("grade_documents", grade_documents)
    graph.add_node("generate", generate)
    graph.add_node("transform_query", transform_query)

    graph.set_entry_point("retrieve")
    graph.add_edge("retrieve", "grade_documents")
    graph.add_conditional_edges(
        "grade_documents",
        decide_to_generate,
        {"generate": "generate", "transform_query": "transform_query"},
    )
    graph.add_edge("transform_query", "retrieve")
    graph.add_edge("generate", END)
    return graph.compile()

corrective_graph = build_self_corrective_graph()
print("Self-Corrective RAG 그래프 빌드 완료")

Self-Corrective RAG 그래프 빌드 완료


---
## 6. 파이프라인 실행

### 6-1. Simple RAG 실행
단순 retrieve → generate 흐름으로 질문에 답변한다.

In [22]:
# Simple RAG 실행
init_state = {
    "question": "이 사업의 총 예산은 얼마인가요?",
    "chat_history": [],
    "filters": {},
    "retrieved_contexts": [],
    "final_answer": "",
    "citations_used": [],
    "errors": [],
    "relevance_score": 0.0,
    "retry_count": 0,
    "latency": 0.0,
    "hit_position": 0,
}

print("=== Simple RAG 실행 ===")
result = simple_graph.invoke(init_state)

print(f"\n질문: {result['question']}")
print(f"\n답변:\n{result['final_answer']}")
print(f"\n출처: {list(dict.fromkeys(result['citations_used']))}")
if result.get("errors"):
    print(f"에러: {result['errors']}")

=== Simple RAG 실행 ===

질문: 이 사업의 총 예산은 얼마인가요?

답변:
요약
- 제공된 RFP 문서들에서 확인되는 사업별 총 예산은 다음과 같습니다.

세부사항
- 농산물가격안정기금 정부예산회계연계시스템 고도화
  - 사업비: 금391,542,840원(부가세 및 회계컨설팅 용역 포함)

- 2025년 통합접수시스템 운영
  - 사업예산: 금738,820,000원(금칠억삼천팔백팔십이만원, VAT포함)

- 2024년 이러닝시스템 운영 용역
  - 제공된 문서 발췌본에서는 총 예산 금액을 확인할 수 없습니다.
  - 따라서 “제공된 문서에서 해당 정보를 확인할 수 없습니다.”

[출처]
- 한국농수산식품유통공사_농산물가격안정기금 정부예산회계연계시스템 .hwp
- 재단법인경기도일자리재단_2025년 통합접수시스템 운영.hwp
- 국민연금공단_2024년 이러닝시스템 운영 용역.hwp

출처: ['한국농수산식품유통공사_농산물가격안정기금 정부예산회계연계시스템 .hwp', '재단법인경기도일자리재단_2025년 통합접수시스템 운영.hwp', '국민연금공단_2024년 이러닝시스템 운영 용역.hwp']


### 6-2. Self-Corrective RAG 실행
관련성 평가 → 재검색 루프가 포함된 고급 흐름.

In [23]:
# Self-Corrective RAG 실행
corrective_state = {
    "question": "입찰 참가 자격 요건은 무엇인가요?",
    "chat_history": [],
    "filters": {},
    "retrieved_contexts": [],
    "final_answer": "",
    "citations_used": [],
    "errors": [],
    "relevance_score": 0.0,
    "retry_count": 0,
    "latency": 0.0,
    "hit_position": 0,
}

print("=== Self-Corrective RAG 실행 ===")
result_sc = corrective_graph.invoke(corrective_state)

print(f"\n질문: {result_sc['question']}")
print(f"관련성 점수: {result_sc['relevance_score']}")
print(f"재시도 횟수: {result_sc['retry_count']}")
print(f"\n답변:\n{result_sc['final_answer']}")
print(f"\n출처: {list(dict.fromkeys(result_sc['citations_used']))}")

=== Self-Corrective RAG 실행 ===
  문서 1: yes
  문서 2: yes
  문서 3: yes
  문서 4: yes
  문서 5: yes
--- 검수 완료: 5/5 관련, 점수 1.00 ---
--- 관련성 충분 → 답변 생성 ---

질문: 입찰 참가 자격 요건은 무엇인가요?
관련성 점수: 1.0
재시도 횟수: 0

답변:
핵심 요약
- 입찰참가자는 "사업금액이 20억원 미만"인 사업의 취지(중소 소프트웨어사업자 지원)에 따라 대기업은 입찰에 참여할 수 없으며(중소 소프트웨어사업자의 사업 참여 지원에 관한 지침 제2조 및 제3조), 지방자치단체 관련 법령·예규에서 정한 자격요건을 모두 충족해야 합니다. 공동수급(컨소시엄)으로 참가할 수 있으나 별도의 조건(구성 수, 지분비율, 중복참가 금지 등)을 준수해야 합니다.

세부 사항
- 기본법적 요건
  - 지방자치단체를 당사자로 하는 계약에 관한 법률 시행령 제13조 및 같은 법 시행규칙 제14조의 자격조건을 갖출 것
  - 같은 법 제31조에 의한 부정당업체로 입찰참가자격의 제한을 받지 아니할 것

- 중소기업·대기업 제한
  - 본 사업은 "사업금액이 20억원 미만"인 사업으로 "중소 소프트웨어사업자의 사업 참여 지원에 관한 지침 제2조 및 제3조"에 따라 대기업은 입찰에 참여할 수 없음

- 필수 업종·증빙(문서에 명시된 모든 항목을 충족해야 함)
  1) 정보통신공사업법 제3조 및 14조에 따른 정보통신공사업(업종코드 0036) 면허를 소지한 등록업체
  2) 소프트웨어진흥법 제58조 및 같은 법 시행령 제53조에 따른 소프트웨어사업자(컴퓨터관련 서비스사업[업종코드 1468]) 제출업체
  3) 소프트웨어사업자의 경우 중소기업제품 구매촉진 및 판로지원에 관한 법률 제9조 및 같은 법 시행령 제10조에 의한 직접생산확인증명서(세부품명 : “정보시스템개발서비스”, 세부품명번호 : 8111159901)를 소지한 자
  4) 직접생산확인증명서(세부품명 : 버스 및 차량정보안내장치, 세부품

---
## 7. 검색 품질 평가

다양한 질문으로 Hit Rate, 응답 시간을 측정한다.

| 지표 | 목표 |
| :--- | :--- |
| Hit Rate @k | >= 0.8 |
| 평균 Latency | < 2초 |

In [24]:
eval_queries = [
    "이 사업의 총 예산은?",
    "제안서 제출 기한이 언제인가요?",
    "입찰 참가 자격 요건은?",
    "사업 범위와 주요 요구사항은?",
    "발주 기관이 어디인가요?",
    "협상에 의한 계약 조건은?",
    "기술 요구사항 중 보안 관련 조건은?",
    "하도급 제한 조건이 있나요?",
]

latencies = []
result_counts = []

print("=== 배치 검색 평가 ===")
for q in eval_queries:
    r = retrieve_for_graph(q)
    n = len(r["retrieved_contexts"])
    lat = r["latency"]
    latencies.append(lat)
    result_counts.append(n)
    status = "OK" if n > 0 else "MISS"
    print(f"  [{status}] {q:30s} -> {n}개, {lat}s")

hit_rate = sum(1 for c in result_counts if c > 0) / len(eval_queries)
avg_latency = sum(latencies) / len(latencies)

print(f"\n=== 평가 결과 ===")
print(f"Hit Rate @{DEFAULT_K}: {hit_rate:.2f} (목표 >= 0.8)")
print(f"평균 Latency: {avg_latency:.4f}s (목표 < 2s)")

=== 배치 검색 평가 ===
  [OK] 이 사업의 총 예산은?                   -> 5개, 0.3445s
  [OK] 제안서 제출 기한이 언제인가요?              -> 5개, 0.3099s
  [OK] 입찰 참가 자격 요건은?                  -> 5개, 0.2973s
  [OK] 사업 범위와 주요 요구사항은?               -> 5개, 0.3433s
  [OK] 발주 기관이 어디인가요?                  -> 5개, 0.3097s
  [OK] 협상에 의한 계약 조건은?                 -> 5개, 0.2964s
  [OK] 기술 요구사항 중 보안 관련 조건은?           -> 5개, 0.286s
  [OK] 하도급 제한 조건이 있나요?                -> 5개, 0.3196s

=== 평가 결과 ===
Hit Rate @5: 1.00 (목표 >= 0.8)
평균 Latency: 0.3133s (목표 < 2s)


In [25]:
# Self-Corrective RAG E2E 평가
print("=== Self-Corrective RAG E2E 평가 ===\n")

eval_qa = [
    "이 사업의 총 예산은 얼마인가요?",
    "제안서 제출 마감일은 언제인가요?",
    "입찰 참가 자격 요건은 무엇인가요?",
]

for q in eval_qa:
    state = {
        "question": q,
        "chat_history": [],
        "filters": {},
        "retrieved_contexts": [],
        "final_answer": "",
        "citations_used": [],
        "errors": [],
        "relevance_score": 0.0,
        "retry_count": 0,
        "latency": 0.0,
        "hit_position": 0,
    }
    start = time.time()
    result = corrective_graph.invoke(state)
    elapsed = time.time() - start

    print(f"Q: {q}")
    print(f"A: {result['final_answer'][:200]}...")
    print(f"출처: {list(dict.fromkeys(result['citations_used']))[:3]}")
    print(f"관련성: {result['relevance_score']:.2f}, 재시도: {result['retry_count']}, 소요: {elapsed:.2f}s")
    print("-" * 60)

=== Self-Corrective RAG E2E 평가 ===

  문서 1: yes
  문서 2: yes
  문서 3: yes
  문서 4: yes
  문서 5: yes
--- 검수 완료: 5/5 관련, 점수 1.00 ---
--- 관련성 충분 → 답변 생성 ---
Q: 이 사업의 총 예산은 얼마인가요?
A: 핵심 요약
- 제공된 문서들 중 예산이 명시된 사업은 두 건이며, 각각 총액은 아래와 같습니다. 일부 문서(이러닝 운영 용역)는 제공된 발췌에서 총예산 정보를 확인할 수 없습니다.

세부 사항
- 농산물가격안정기금 정부예산회계연계시스템 고도화
  - 사업비: 금391,542,840원(부가세 및 회계컨설팅 용역 포함)
  - (계약일로부터 7개월)
- 202...
출처: ['한국농수산식품유통공사_농산물가격안정기금 정부예산회계연계시스템 .hwp', '재단법인경기도일자리재단_2025년 통합접수시스템 운영.hwp', '국민연금공단_2024년 이러닝시스템 운영 용역.hwp']
관련성: 1.00, 재시도: 0, 소요: 41.06s
------------------------------------------------------------
  문서 1: no
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: yes
--- 검수 완료: 1/5 관련, 점수 0.20 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 1) ---
  원래: 제안서 제출 마감일은 언제인가요?
  변환: 제안서 제출기한(마감일)은?
  문서 1: no','description':'문서에 제안서 제출기한이 구체적으로 명시되어 있지 않고 "입찰공고에 따름"으로만 되어 있어 질문에 대한 직접적인 답을 제공하지 않습니다.
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: no
--- 검수 완료: 0/5 관련, 점수 0.00 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 2) ---
  

---
## 8. 질의응답 (Q&A)

100개 RFP 문서 전체를 대상으로 질의응답을 수행한다.

### 사용법
```python
# 기본 질문
ask("이 사업의 총 예산은?")

# 특정 기관 필터
ask("사업 개요를 알려줘", filters={"기관": "국방부"})

# 특정 파일 대상
ask("참가 자격은?", filters={"파일": "파일명.hwp"})
```

In [26]:
# === ask() 편의 함수 ===

def ask(question: str, filters: dict = None, mode: str = "corrective") -> dict:
    """100개 RFP 문서를 대상으로 질의응답을 수행한다.

    Args:
        question: 사용자 질문
        filters: 검색 필터 (선택). {"기관": "...", "파일": "..."} 형태
        mode: "simple" (단순 RAG) 또는 "corrective" (Self-Corrective RAG)

    Returns:
        dict with keys: answer, sources, relevance, retries, elapsed
    """
    state = {
        "question": question,
        "chat_history": [],
        "filters": filters or {},
        "retrieved_contexts": [],
        "final_answer": "",
        "citations_used": [],
        "errors": [],
        "relevance_score": 0.0,
        "retry_count": 0,
        "latency": 0.0,
        "hit_position": 0,
    }

    graph = corrective_graph if mode == "corrective" else simple_graph
    start = time.time()
    result = graph.invoke(state)
    elapsed = round(time.time() - start, 2)

    # 중복 제거된 출처 목록
    sources = list(dict.fromkeys(result.get("citations_used", [])))

    # 출력
    print(f"Q: {question}")
    if filters:
        print(f"   필터: {filters}")
    print(f"\nA: {result['final_answer']}")
    print(f"\n[출처] {', '.join(sources[:5])}")
    if mode == "corrective":
        print(f"[관련성] {result.get('relevance_score', 0):.2f}  [재시도] {result.get('retry_count', 0)}  [소요] {elapsed}s")
    else:
        print(f"[소요] {elapsed}s")

    if result.get("errors"):
        print(f"[경고] {result['errors']}")

    return {
        "answer": result["final_answer"],
        "sources": sources,
        "relevance": result.get("relevance_score", 0),
        "retries": result.get("retry_count", 0),
        "elapsed": elapsed,
        "errors": result.get("errors", []),
    }


print("ask() 함수 정의 완료")

ask() 함수 정의 완료


### 8-1. 데이터셋 현황

벡터스토어에 적재된 문서와 발주기관 목록을 확인한다.

In [27]:
# 적재된 데이터셋 현황 확인
print("=== 벡터스토어 현황 ===")
print(f"총 벡터 수: {vectorstore._collection.count()}")
print(f"문서 수: {len(df_raw)}")

# 발주기관 목록
agencies = df_raw["발주 기관"].dropna().unique()
print(f"\n=== 발주기관 목록 ({len(agencies)}개) ===")
for i, agency in enumerate(sorted(agencies), 1):
    print(f"  {i:2d}. {agency}")

# 파일 목록 (상위 10개)
files = df_raw["파일명"].tolist()
print(f"\n=== 파일 목록 (상위 10개 / 전체 {len(files)}개) ===")
for f in files[:10]:
    print(f"  - {f}")
if len(files) > 10:
    print(f"  ... 외 {len(files) - 10}개")

=== 벡터스토어 현황 ===
총 벡터 수: 2455
문서 수: 100

=== 발주기관 목록 (87개) ===
   1. (사)벤처기업협회
   2. (사)부산국제영화제
   3. (사）한국대학스포츠협의회
   4. (재)예술경영지원센터
   5. 2025 구미 아시아육상경기선수권대회 조직위원회
   6. BioIN
   7. KOICA 전자조달
   8. 경기도 안양시
   9. 경기도 평택시
  10. 경기도사회서비스원
  11. 경상북도 봉화군
  12. 경희대학교
  13. 고려대학교
  14. 고양도시관리공사
  15. 광주과학기술원
  16. 국가과학기술지식정보서비스
  17. 국가철도공단
  18. 국립인천해양박물관
  19. 국립중앙의료원
  20. 국민연금공단
  21. 국방과학연구소
  22. 그랜드코리아레저(주)
  23. 기초과학연구원
  24. 나노종합기술원
  25. 남서울대학교
  26. 대검찰청
  27. 대전대학교
  28. 대한상공회의소
  29. 대한장애인체육회
  30. 대한적십자사 의료원
  31. 문화체육관광부 국립민속박물관
  32. 부산관광공사
  33. 사단법인 보험개발원
  34. 사단법인아시아물위원회사무국
  35. 서민금융진흥원
  36. 서영대학교 산학협력단
  37. 서울시립대학교
  38. 서울특별시
  39. 서울특별시 여성가족재단
  40. 서울특별시교육청
  41. 세종테크노파크
  42. 수협중앙회
  43. 울산광역시
  44. 을지대학교
  45. 인천공항운영서비스(주)
  46. 인천광역시
  47. 인천광역시 동구
  48. 재단법인 광주광역시 광주문화재단
  49. 재단법인 광주연구원
  50. 재단법인 한국장애인문화예술원
  51. 재단법인경기도일자리재단
  52. 재단법인스포츠윤리센터
  53. 재단법인충북연구원
  54. 전북대학교
  55. 전북특별자치도 정읍시
  56. 조선대학교
  57. 중앙선거관리위원회
  58. 축산물품질평가원
  59. 케빈랩 주식회사
  60. 파주도

### 8-2. 질의응답 예시

다양한 질문 유형으로 100개 RFP 문서에 대한 Q&A를 시연한다.

In [28]:
# 예시 1: 첫 번째 데이터(한영대학교) — 사업 개요 질문
ask("한영대학교 특성화 맞춤형 교육환경 구축 사업의 개요를 알려줘",
    filters={"파일": "한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp"})

  문서 1: yes
  문서 2: yes
  문서 3: yes
  문서 4: yes
  문서 5: yes
--- 검수 완료: 5/5 관련, 점수 1.00 ---
--- 관련성 충분 → 답변 생성 ---
Q: 한영대학교 특성화 맞춤형 교육환경 구축 사업의 개요를 알려줘
   필터: {'파일': '한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp'}

A: 핵심 요약
- 한영대학교의 특성화 맞춤형 교육환경 구축 사업으로, "트랙운영 학사정보시스템 고도화"를 통해 트랙제 기반의 교육과정 운영·관리 체계를 개선하고 산업체 연계형 교육 강화를 지원하는 사업입니다. 사업예산은 "130,000,000원 범위 내 (VAT 포함)"이며, 입찰은 "제한경쟁입찰(협상에 의한 계약 체결)" 방식입니다.

세부 사항
- 사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화
- 발주기관: 한영대학
- 공고번호: 20241001798
- 사업예산: 130,000,000원 범위 내 (VAT 포함)
- 입찰/계약방법: 제한경쟁입찰(협상에 의한 계약 체결)
- 제출기한: 2024-10-15 17:00:00
- 사업기간: 계약일로부터 3개월 (안정화기간 1개월 포함)
  - 비고: 기간 및 일정은 학교 사정과 용역대상자와의 협의에 따라 조정될 수 있음
- 사업목적 / 요약 내용:
  - 트랙운영 학사정보시스템을 고도화하여 트랙제도 도입을 지원
  - 전공교과목 선택폭 확대를 통해 다양한 진로선택 기회 제공 및 취업문 확대
  - 트랙제 교육과정 운영으로 산업현장 경쟁력 강화 및 산업체 수요 맞춤 교육과정 운영 활성화
  - 트랙기반 교육과정의 운영 및 관리 체계를 효과적으로 지원
  - 교수자-학습자 중심의 교육환경 조성 및 대학 체제 개편에 대한 대응체계 확립
- 기타: 문서 목차에는 구축목표·구축일정·구축범위·제안요청 내용·제안서 작성 및 평가·제출서류 등 제안 관련 상세 안내 항목이 포함되어 있음

[출처]
- 한영대학_한영대학교 

{'answer': '핵심 요약\n- 한영대학교의 특성화 맞춤형 교육환경 구축 사업으로, "트랙운영 학사정보시스템 고도화"를 통해 트랙제 기반의 교육과정 운영·관리 체계를 개선하고 산업체 연계형 교육 강화를 지원하는 사업입니다. 사업예산은 "130,000,000원 범위 내 (VAT 포함)"이며, 입찰은 "제한경쟁입찰(협상에 의한 계약 체결)" 방식입니다.\n\n세부 사항\n- 사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화\n- 발주기관: 한영대학\n- 공고번호: 20241001798\n- 사업예산: 130,000,000원 범위 내 (VAT 포함)\n- 입찰/계약방법: 제한경쟁입찰(협상에 의한 계약 체결)\n- 제출기한: 2024-10-15 17:00:00\n- 사업기간: 계약일로부터 3개월 (안정화기간 1개월 포함)\n  - 비고: 기간 및 일정은 학교 사정과 용역대상자와의 협의에 따라 조정될 수 있음\n- 사업목적 / 요약 내용:\n  - 트랙운영 학사정보시스템을 고도화하여 트랙제도 도입을 지원\n  - 전공교과목 선택폭 확대를 통해 다양한 진로선택 기회 제공 및 취업문 확대\n  - 트랙제 교육과정 운영으로 산업현장 경쟁력 강화 및 산업체 수요 맞춤 교육과정 운영 활성화\n  - 트랙기반 교육과정의 운영 및 관리 체계를 효과적으로 지원\n  - 교수자-학습자 중심의 교육환경 조성 및 대학 체제 개편에 대한 대응체계 확립\n- 기타: 문서 목차에는 구축목표·구축일정·구축범위·제안요청 내용·제안서 작성 및 평가·제출서류 등 제안 관련 상세 안내 항목이 포함되어 있음\n\n[출처]\n- 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp',
 'sources': ['한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp'],
 'relevance': 1.0,
 'retries': 0,
 'elapsed': 57.1,
 'errors': []}

In [29]:
# 예시 2: 첫 번째 데이터 — 예산 질문
ask("이 사업의 총 예산은 얼마인가요?",
    filters={"파일": "한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp"})

  문서 1: no
  문서 2: no
  문서 3: no
  문서 4: yes
  문서 5: yes
--- 검수 완료: 2/5 관련, 점수 0.40 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 1) ---
  원래: 이 사업의 총 예산은 얼마인가요?
  변환: 사업 총예산은 얼마? 추정가격·총액
  문서 1: no
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: yes
--- 검수 완료: 1/5 관련, 점수 0.20 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 2) ---
  원래: 사업 총예산은 얼마? 추정가격·총액
  변환: 사업 총예산(추정가격·총액)?
  문서 1: no
  문서 2: no','description':'문서 관련성 판별 결과.
  문서 3: no
  문서 4: no
  문서 5: yes
--- 검수 완료: 1/5 관련, 점수 0.20 ---
--- 최대 재시도(2회) 도달 → 답변 생성 ---
Q: 이 사업의 총 예산은 얼마인가요?
   필터: {'파일': '한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp'}

A: 핵심 요약
- 사업 총예산(추정가격·총액)은 "사업예산 : 130,000,000원 범위 내 (VAT 포함)"입니다.

세부 사항
- 원문 표기: 사업예산 : 130,000,000원 범위 내 (VAT 포함)
- 해당 금액은 문서에 "범위 내"로 명시되어 있으므로, 상기 한도 내에서 예산이 책정됨을 의미합니다.

[출처]
- 문서 5: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp (목차 및 사업개요)

[출처] 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp
[관련성] 0.20  [재시도] 2  [소요] 102.7s


{'answer': '핵심 요약\n- 사업 총예산(추정가격·총액)은 "사업예산 : 130,000,000원 범위 내 (VAT 포함)"입니다.\n\n세부 사항\n- 원문 표기: 사업예산 : 130,000,000원 범위 내 (VAT 포함)\n- 해당 금액은 문서에 "범위 내"로 명시되어 있으므로, 상기 한도 내에서 예산이 책정됨을 의미합니다.\n\n[출처]\n- 문서 5: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp (목차 및 사업개요)',
 'sources': ['한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp'],
 'relevance': 0.2,
 'retries': 2,
 'elapsed': 102.7,
 'errors': []}

In [30]:
# 예시 3: 첫 번째 데이터 — 제출 기한 질문
ask("제안서 제출 마감일은 언제인가요?",
    filters={"파일": "한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp"})

  문서 1: no
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: no
--- 검수 완료: 0/5 관련, 점수 0.00 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 1) ---
  원래: 제안서 제출 마감일은 언제인가요?
  변환: 제안서 제출 마감일(제출기한/마감시한)?
  문서 1: no','description':'문서 관련성 판결 결과.
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: no
--- 검수 완료: 0/5 관련, 점수 0.00 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 2) ---
  원래: 제안서 제출 마감일(제출기한/마감시한)?
  변환: 제안서 제출마감일/제출기한?
  문서 1: no
  문서 2: no
  문서 3: no","description":"문서 관련성 판별 결과."} Invalid JSON—there's an error: the assistant produced malformed JSON (an extra quote). The assistant must output valid JSON only. Please correct and resend. The corrected JSON should follow the schema exactly. Please respond again with a valid JSON object containing only the fields 
  문서 4: no
  문서 5: no
--- 검수 완료: 0/5 관련, 점수 0.00 ---
--- 최대 재시도(2회) 도달 → 답변 생성 ---
Q: 제안서 제출 마감일은 언제인가요?
   필터: {'파일': '한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp'}

A: 요약
- 제공된 문서들에서 제안서 제출마감일(제출기한)은 확인할 수 없습니다.

세부사항
- 문서 목차에 "제안서 제출일정 및 방법 - 1

{'answer': '요약\n- 제공된 문서들에서 제안서 제출마감일(제출기한)은 확인할 수 없습니다.\n\n세부사항\n- 문서 목차에 "제안서 제출일정 및 방법 - 17" 항목이 있으나, 본문(페이지 17)의 구체적 제출일·시간 내용은 제공된 발췌문에 포함되어 있지 않습니다.\n- 참고 가능한 공고 내 기재사항(원문 그대로 인용):\n  - 사업예산 : 130,000,000원 범위 내 (VAT 포함)\n  - 사업기간 : 계약일로부터 3개월 (안정화기간 1개월 포함)\n  - 입찰방법 : 제한경쟁입찰(협상에 의한 계약 체결)\n  - "* 기간 및 일정은 학교 사정과 용역대상자와의 협의에 따라 조정될 수 있음" (원문 문구)\n- 제출마감일을 확인하려면 원문 RFP의 "제안서 제출일정 및 방법"(페이지 17) 항목 또는 발주기관(한영대학교)으로 직접 문의하시기 바랍니다.\n\n[출처]\n- 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp (제공된 발췌문)',
 'sources': ['한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp'],
 'relevance': 0.0,
 'retries': 2,
 'elapsed': 122.02,
 'errors': []}

In [31]:
# 예시 4: 첫 번째 데이터 — 참가 자격 질문
ask("입찰 참가 자격 요건은 무엇인가요?",
    filters={"파일": "한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp"})

  문서 1: no
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: no
--- 검수 완료: 0/5 관련, 점수 0.00 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 1) ---
  원래: 입찰 참가 자격 요건은 무엇인가요?
  변환: 입찰 참가 자격요건·참여자격
  문서 1: no
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: no
--- 검수 완료: 0/5 관련, 점수 0.00 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 2) ---
  원래: 입찰 참가 자격요건·참여자격
  변환: 입찰 참가 자격요건(참여자격, 자격조건)
  문서 1: no
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: no
--- 검수 완료: 0/5 관련, 점수 0.00 ---
--- 최대 재시도(2회) 도달 → 답변 생성 ---
Q: 입찰 참가 자격 요건은 무엇인가요?
   필터: {'파일': '한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp'}

A: 핵심 요약
- 제공된 문서들에서는 입찰 참가 자격요건(참여자격, 자격조건)의 구체 항목을 확인할 수 없습니다. 다만 입찰방식·예산·제출기한·사업기간 등 기본 정보는 문서에 명시되어 있습니다.

세부사항
- 입찰 참가 자격요건(참여자격·자격조건)
  - 제공된 문서에서 해당 정보를 확인할 수 없습니다.
- 문서에 명시된 관련 기본 정보(참고)
  - 사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화
  - 발주기관: 한영대학
  - 사업예산: "130,000,000원 범위 내 (VAT 포함)" (문서 5)
  - 입찰방식/방법:
    - 문서 2/3/4: "입찰방식: 협상에 의한 계약"
    - 문서 5: "입찰방법 : 제한경쟁입찰(협상에 의한 계약 체결)"
  - 제출기한: "2024-10-15 17:00:00" (문서 

{'answer': '핵심 요약\n- 제공된 문서들에서는 입찰 참가 자격요건(참여자격, 자격조건)의 구체 항목을 확인할 수 없습니다. 다만 입찰방식·예산·제출기한·사업기간 등 기본 정보는 문서에 명시되어 있습니다.\n\n세부사항\n- 입찰 참가 자격요건(참여자격·자격조건)\n  - 제공된 문서에서 해당 정보를 확인할 수 없습니다.\n- 문서에 명시된 관련 기본 정보(참고)\n  - 사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화\n  - 발주기관: 한영대학\n  - 사업예산: "130,000,000원 범위 내 (VAT 포함)" (문서 5)\n  - 입찰방식/방법:\n    - 문서 2/3/4: "입찰방식: 협상에 의한 계약"\n    - 문서 5: "입찰방법 : 제한경쟁입찰(협상에 의한 계약 체결)"\n  - 제출기한: "2024-10-15 17:00:00" (문서 2/3/4)\n  - 사업기간: "계약일로부터 3개월 (안정화기간 1개월 포함)" (문서 5)\n  - 공고번호: 20241001798 (문서 2/3/4)\n- 추가 안내\n  - 본 RFP의 목차(문서 5)는 "Ⅳ. 제안안내 사항" 하위에 "1. 입찰 및 계약방법", "5. 제출서류" 등이 있음을 보여주나, 제공된 발췌본에는 해당 항목들의 세부 내용(참여자격·자격조건 포함)이 포함되어 있지 않습니다. 해당 세부 조건 확인을 위해서는 원문(Ⅳ절 관련 페이지)의 전체 내용을 확인해야 합니다.\n\n[출처]\n- 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp (문서 1~5 제공 자료)',
 'sources': ['한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp'],
 'relevance': 0.0,
 'retries': 2,
 'elapsed': 135.64,
 'errors': []}

In [32]:
# 예시 5: 첫 번째 데이터 — 기술 요구사항 질문
ask("주요 기술 요구사항은 무엇인가요?",
    filters={"파일": "한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp"})

  문서 1: yes
  문서 2: yes
  문서 3: yes
  문서 4: yes
  문서 5: yes
--- 검수 완료: 5/5 관련, 점수 1.00 ---
--- 관련성 충분 → 답변 생성 ---
Q: 주요 기술 요구사항은 무엇인가요?
   필터: {'파일': '한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp'}

A: 핵심 요약
- 문서상 명시된 주요 기술 요구사항은 트랙제 기반 교육과정을 “교과과정 개발에서 성적 이수까지” 전주기를 지원하도록 시스템을 구현하고, 현행 종합정보시스템(교육과정 / 수강신청 / 성적 / 학적관리 등)과 연계할 수 있도록 고도화하며, 확장성·유연성·유지보수 용이성 및 교수·직원·학생 등 다양한 사용자의 정보 접근성과 사용 편의성을 고려하는 것입니다.
- 사업 핵심 수치(예산·기간)는 문서에 다음과 같이 명시되어 있습니다: 사업예산 : 130,000,000원 범위 내 (VAT 포함), 사업기간 : 계약일로부터 3개월 (안정화기간 1개월 포함).

세부 기술 요구사항 (문서 통합 정리)
- 기능 범위(시스템 구현 범위)
  - 트랙제도 교과과정 개편 및 표준운영관리를 위한 시스템 구현(교과과정 개발 → 성적 이수까지 전주기 지원)
- 연계·고도화 요구
  - 트랙제도 기반 교육과정과 현행 종합정보시스템의 연계 가능하도록 고도화
  - 연계 대상 예시(문서 표기): 교육과정 / 수강신청 / 성적 / 학적관리
- 비기능(품질) 요구
  - 확장성과 유연성을 고려한 환경 조성
  - 유지보수가 용이하도록 설계·구현
  - 다양한 사용자(교수, 직원, 학생)의 정보 접근성 및 사용 편의성 고려(사용자 중심 UI/UX 관점)
- 사업제약·일정·예산 관련 요구
  - 사업예산 : 130,000,000원 범위 내 (VAT 포함)
  - 구축기간 : 계약일로부터 3개월 (안정화기간 1개월 포함)

문서에 명시되어 있지 않거나 확인 불가한 기술 항목
- 인터페이스 규격(예: API 형태, 데이터 포맷, 표준 연계 

{'answer': '핵심 요약\n- 문서상 명시된 주요 기술 요구사항은 트랙제 기반 교육과정을 “교과과정 개발에서 성적 이수까지” 전주기를 지원하도록 시스템을 구현하고, 현행 종합정보시스템(교육과정 / 수강신청 / 성적 / 학적관리 등)과 연계할 수 있도록 고도화하며, 확장성·유연성·유지보수 용이성 및 교수·직원·학생 등 다양한 사용자의 정보 접근성과 사용 편의성을 고려하는 것입니다.\n- 사업 핵심 수치(예산·기간)는 문서에 다음과 같이 명시되어 있습니다: 사업예산 : 130,000,000원 범위 내 (VAT 포함), 사업기간 : 계약일로부터 3개월 (안정화기간 1개월 포함).\n\n세부 기술 요구사항 (문서 통합 정리)\n- 기능 범위(시스템 구현 범위)\n  - 트랙제도 교과과정 개편 및 표준운영관리를 위한 시스템 구현(교과과정 개발 → 성적 이수까지 전주기 지원)\n- 연계·고도화 요구\n  - 트랙제도 기반 교육과정과 현행 종합정보시스템의 연계 가능하도록 고도화\n  - 연계 대상 예시(문서 표기): 교육과정 / 수강신청 / 성적 / 학적관리\n- 비기능(품질) 요구\n  - 확장성과 유연성을 고려한 환경 조성\n  - 유지보수가 용이하도록 설계·구현\n  - 다양한 사용자(교수, 직원, 학생)의 정보 접근성 및 사용 편의성 고려(사용자 중심 UI/UX 관점)\n- 사업제약·일정·예산 관련 요구\n  - 사업예산 : 130,000,000원 범위 내 (VAT 포함)\n  - 구축기간 : 계약일로부터 3개월 (안정화기간 1개월 포함)\n\n문서에 명시되어 있지 않거나 확인 불가한 기술 항목\n- 인터페이스 규격(예: API 형태, 데이터 포맷, 표준 연계 방법), 인증·권한·보안 요구사항, 성능(응답시간·동시접속자 수) 목표, 대상 플랫폼(OS/DB/웹서버/모바일), 백업·복구·장애대응(SLA), 데이터이관(마이그레이션) 범위 및 방식, 테스트 요구(단위/통합/사용자수용), 운영·유지보수 기간·조건, 문서화·교육 요구사항 등은 제공된 문서에서 해당 정보

### 8-3. 발주기관 필터 질의

특정 발주기관 소속 문서만 대상으로 질의한다.

In [33]:
# 예시 6: 발주기관 필터 — 한영대학 사업 요약
ask("사업 범위와 주요 요구사항을 정리해줘", filters={"기관": "한영대학"})

  문서 1: yes
  문서 2: yes
  문서 3: yes
  문서 4: yes
  문서 5: yes
--- 검수 완료: 5/5 관련, 점수 1.00 ---
--- 관련성 충분 → 답변 생성 ---
Q: 사업 범위와 주요 요구사항을 정리해줘
   필터: {'기관': '한영대학'}

A: 핵심 요약
- 사업명은 "한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화"이며, 목표는 트랙제 기반 교육과정의 개편·운영을 지원하는 학사정보시스템을 고도화하는 것입니다. 주요 요구사항은 트랙제 교육과정의 전주기(교과과정 개발 → 수강신청 → 성적 → 학적관리 등)를 지원하고, 현행 종합정보시스템과 연계 가능하도록 구현하는 것, 그리고 확장성·유연성·유지보수성·사용편의성을 확보하는 것입니다. (예산 및 기간 등 핵심 수치는 아래에 원문 그대로 제시)

사업 범위 (문서에 명시된 내용 기준)
- 사업명
  - "한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화"
- 예산
  - "사업예산 : 130,000,000원 범위 내 (VAT 포함)"
- 사업기간 / 일정
  - "구축기간 : 계약일로부터 3개월 (안정화기간 1개월 포함)"
  - 문서에 W1~W12로 구성된 구축·안정화 일정표(주 단위)가 제시되어 있음
- 주요 범위(연계 대상 등)
  - "트랙제도 교과과정 개편 및 표준운영관리를 위한 시스템 구현 (교과과정 개발에서 성적 이수까지)"
  - 트랙제도 기반 교육과정과 현행 종합정보시스템 연계 고도화:
    - 연계 대상 예시로 문서에 기재된 항목: "(교육과정 / 수강신청 / 성적 / 학적관리 외)"
- 입찰/계약 방식(관련)
  - "입찰방법 : 제한경쟁입찰(협상에 의한 계약 체결)"

주요 요구사항 (문서에 명시된 항목 기준)
- 기능·운영 목표
  - 트랙제도 교과과정의 개편 및 표준 운영 관리를 위한 시스템 구현(교과과정 개발에서 성적 이수까지)
  - 트랙기반 교육과정의 운영 및 관리 체계를 효과적으로 지원


{'answer': '핵심 요약\n- 사업명은 "한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화"이며, 목표는 트랙제 기반 교육과정의 개편·운영을 지원하는 학사정보시스템을 고도화하는 것입니다. 주요 요구사항은 트랙제 교육과정의 전주기(교과과정 개발 → 수강신청 → 성적 → 학적관리 등)를 지원하고, 현행 종합정보시스템과 연계 가능하도록 구현하는 것, 그리고 확장성·유연성·유지보수성·사용편의성을 확보하는 것입니다. (예산 및 기간 등 핵심 수치는 아래에 원문 그대로 제시)\n\n사업 범위 (문서에 명시된 내용 기준)\n- 사업명\n  - "한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화"\n- 예산\n  - "사업예산 : 130,000,000원 범위 내 (VAT 포함)"\n- 사업기간 / 일정\n  - "구축기간 : 계약일로부터 3개월 (안정화기간 1개월 포함)"\n  - 문서에 W1~W12로 구성된 구축·안정화 일정표(주 단위)가 제시되어 있음\n- 주요 범위(연계 대상 등)\n  - "트랙제도 교과과정 개편 및 표준운영관리를 위한 시스템 구현 (교과과정 개발에서 성적 이수까지)"\n  - 트랙제도 기반 교육과정과 현행 종합정보시스템 연계 고도화:\n    - 연계 대상 예시로 문서에 기재된 항목: "(교육과정 / 수강신청 / 성적 / 학적관리 외)"\n- 입찰/계약 방식(관련)\n  - "입찰방법 : 제한경쟁입찰(협상에 의한 계약 체결)"\n\n주요 요구사항 (문서에 명시된 항목 기준)\n- 기능·운영 목표\n  - 트랙제도 교과과정의 개편 및 표준 운영 관리를 위한 시스템 구현(교과과정 개발에서 성적 이수까지)\n  - 트랙기반 교육과정의 운영 및 관리 체계를 효과적으로 지원\n  - 교수자-학습자 중심의 교육환경 조성\n  - 학사운영 시스템을 통해 대학 체제 개편에 대한 대응체계 확립\n- 시스템 통합 및 고도화\n  - 현행 종합정보시스템과의 연계 가능성 확보(교육과정, 수강신청, 성적, 학적관리 등)

### 8-4. 필터 없이 전체 문서 대상 질의

100개 문서 전체를 대상으로 질의하여 여러 사업 정보를 통합 답변받는다.

In [34]:
# 예시 7: 전체 문서 대상 — 협상에 의한 계약 조건
ask("협상에 의한 계약 방식을 사용하는 사업은 어떤 것이 있나요?")

  문서 1: yes
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: no
--- 검수 완료: 1/5 관련, 점수 0.20 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 1) ---
  원래: 협상에 의한 계약 방식을 사용하는 사업은 어떤 것이 있나요?
  변환: 협상계약 대상 사업? 입찰유형, 계약절차
  문서 1: yes
  문서 2: yes
  문서 3: yes
  문서 4: yes
  문서 5: yes
--- 검수 완료: 5/5 관련, 점수 1.00 ---
--- 관련성 충분 → 답변 생성 ---
Q: 협상에 의한 계약 방식을 사용하는 사업은 어떤 것이 있나요?

A: 핵심 요약
- 제공된 문서들에 따르면 해당 사업들은 모두 "협상에 의한 계약" 방식이 적용됩니다. 다만 입찰유형은 사업마다 다르며, 울산 버스정보시스템 문서에서는 "제한경쟁입찰", 다른(ODA) 문서에서는 "일반경쟁(협상에 의한 계약)"으로 명시되어 있습니다.
- 계약절차는 기술제안서 제출 → 기술평가 → 우선협상대상자 선정 → 기술협상 실시 → 계약체결 → 사업착수 순으로 진행됩니다. (문서별로 세부 요건·선정기준이 추가로 규정되어 있음)

세부 사항
- 협상계약 대상 여부
  - 울산 버스정보시스템 관련 문서(문서 1,3,4,5): "계약방식 : 협상에 의한 계약체결"
  - ODA 관련 문서(문서 2): 입찰방법으로 "일반경쟁(협상에 의한 계약)" 명시

- 입찰유형(문서별)
  - 울산 버스정보시스템: "입찰방식 : 제한경쟁입찰"
  - ODA 사업(문서 2): "입찰방법 : 일반경쟁(협상에 의한 계약)"

- 계약절차 및 흐름(공통)
  - 제시된 흐름: 입 찰 공 고 및 규격공개 → 기술제안서 작성·제출 → 기술제안서 평가 → 평가 결과 통보 → 심사지적사항 확인 및 발주기관 의견 정리 → 우선협상대상자 선정 → 기술협상 실시(발주기관, 우선협상 대상자) → 계약체결(발주기관, 계약상대자) → 사업착수 → 사업시행 →

{'answer': '핵심 요약\n- 제공된 문서들에 따르면 해당 사업들은 모두 "협상에 의한 계약" 방식이 적용됩니다. 다만 입찰유형은 사업마다 다르며, 울산 버스정보시스템 문서에서는 "제한경쟁입찰", 다른(ODA) 문서에서는 "일반경쟁(협상에 의한 계약)"으로 명시되어 있습니다.\n- 계약절차는 기술제안서 제출 → 기술평가 → 우선협상대상자 선정 → 기술협상 실시 → 계약체결 → 사업착수 순으로 진행됩니다. (문서별로 세부 요건·선정기준이 추가로 규정되어 있음)\n\n세부 사항\n- 협상계약 대상 여부\n  - 울산 버스정보시스템 관련 문서(문서 1,3,4,5): "계약방식 : 협상에 의한 계약체결"\n  - ODA 관련 문서(문서 2): 입찰방법으로 "일반경쟁(협상에 의한 계약)" 명시\n\n- 입찰유형(문서별)\n  - 울산 버스정보시스템: "입찰방식 : 제한경쟁입찰"\n  - ODA 사업(문서 2): "입찰방법 : 일반경쟁(협상에 의한 계약)"\n\n- 계약절차 및 흐름(공통)\n  - 제시된 흐름: 입 찰 공 고 및 규격공개 → 기술제안서 작성·제출 → 기술제안서 평가 → 평가 결과 통보 → 심사지적사항 확인 및 발주기관 의견 정리 → 우선협상대상자 선정 → 기술협상 실시(발주기관, 우선협상 대상자) → 계약체결(발주기관, 계약상대자) → 사업착수 → 사업시행 → 시험운영 후 준공\n  - 사업기간 표기(울산 문서): "사업시행 4개월", "시험운영 후 준공 1개월"\n\n- 선정 기준·절차상 주요 규정 (문서 2 등)\n  - 기술평가 기준 관련: "기술능력평가분야 배점한도의 85% 이상인 자를 협상적격자로 선정"\n    - "협상적격자가 1인의 경우에도 유효함"\n  - 공동수급(공동도급) 규칙: "공동도급 총 업체수는 대표사를 포함하여 5개사 이하, 공동수급으로 참여하는 업체의 최소지분율은 10% 이상이어야 함"\n    - "대표사는 등록업체 중 이행비율(지분율)이 가장 큰 회사가 되어야 하며, 사업책임기술자(PM)은 대표사 소속이어야 함"

### 8-5. 대화형 Q&A

`input()`을 통해 자유롭게 질문할 수 있다. `q` 또는 `종료`를 입력하면 종료된다.

In [35]:
# 대화형 Q&A 루프
print("=" * 60)
print("  입찰메이트 Q&A — 100개 RFP 문서 질의응답")
print("  종료: 'q' 또는 '종료' 입력")
print("  필터 사용: '필터:기관=한영대학' 형태로 질문 앞에 추가")
print("=" * 60)

while True:
    user_input = input("\n질문> ").strip()
    if not user_input or user_input.lower() in ("q", "quit", "exit", "종료"):
        print("Q&A를 종료합니다.")
        break

    # 필터 파싱: "필터:기관=한영대학 질문내용" 형태 지원
    filters = {}
    question = user_input
    if user_input.startswith("필터:"):
        parts = user_input.split(" ", 1)
        filter_str = parts[0].replace("필터:", "")
        question = parts[1] if len(parts) > 1 else ""
        for pair in filter_str.split(","):
            if "=" in pair:
                key, val = pair.split("=", 1)
                filters[key.strip()] = val.strip()

    if not question:
        print("질문을 입력해주세요.")
        continue

    print("-" * 60)
    ask(question, filters=filters if filters else None)
    print("-" * 60)

  입찰메이트 Q&A — 100개 RFP 문서 질의응답
  종료: 'q' 또는 '종료' 입력
  필터 사용: '필터:기관=한영대학' 형태로 질문 앞에 추가
------------------------------------------------------------
  문서 1: no
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: no
--- 검수 완료: 0/5 관련, 점수 0.00 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 1) ---
  원래: 요구하는 기술스택을 알려줘
  변환: 요구 기술스택·필수사양
  문서 1: no
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: yes
--- 검수 완료: 1/5 관련, 점수 0.20 ---
--- 관련성 부족 → 질문 재작성 ---
--- 질문 재작성 (retry 2) ---
  원래: 요구 기술스택·필수사양
  변환: 요구 기술스택·필수사양(기술요건, 제출기준)
  문서 1: no
  문서 2: no
  문서 3: no
  문서 4: no
  문서 5: no
--- 검수 완료: 0/5 관련, 점수 0.00 ---
--- 최대 재시도(2회) 도달 → 답변 생성 ---
Q: 요구하는 기술스택을 알려줘

A: 핵심 요약
- 제공된 문서들에서는 구체적인 기술스택(예: 프로그래밍 언어, 프레임워크, DBMS 등), 세부 하드웨어·서버 사양(예: CPU, 메모리, 저장공간, OS 버전), 또는 제출용 템플릿/포맷 같은 "요구 기술스택·필수사양(기술요건, 제출기준)"을 명시적으로 확인할 수 없습니다. (제공된 문서에서 해당 정보를 확인할 수 없습니다.)
- 다만 요구사항 분류와 항목 수, ECR(시스템 장비 구성요구사항) 등 관련 항목의 존재와 평가 관련 기준은 문서에 기술되어 있습니다.

세부 사항
- 기술스택/필수사양 관련 정보 부재
  - 제공된 문서들에서는 구체적인 기술스택(프로그래밍 언어/프레임워크/미들웨어/DBMS 등) 또는 